# Evaluación de M0 y M1 sobre el test real (lote 1)

Compara el motor de reglas (M0) y el clasificador TF-IDF + regresión logística (M1) con las mismas preguntas reales de prueba.

- M0 no se entrena. M1 se entrena en tres variantes: solo con el *train* de la semilla sintética, solo con el *train* real y con ambos.
- El hiperparámetro `C` y el umbral de confianza de M1 se eligen dentro de `fit()` por validación cruzada sobre el *train* de cada variante. El *test* nunca participa en esa elección.
- Se evalúa **una sola vez** sobre el *test* real. Si `reports/results_fase2.csv` ya existe, la celda de corrida se niega a repetirse.
- Métricas: accuracy de intención, abstención correcta, abstención innecesaria y latencia, con intervalos de confianza (IC) bootstrap al 95 %.

Ejecutar desde `notebooks/`, con el entorno virtual del proyecto y de arriba hacia abajo.

In [6]:
import platform
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

try:
    from IPython.display import display
except ImportError:  # ejecución fuera de Jupyter
    display = print

# El notebook vive en notebooks/; la raíz del repositorio está un nivel arriba.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import ESCALATING_INTENTS, INTENTS, PROCESSED_DIR, REPORTS_DIR, SEED  # noqa: E402
from src.engines.m1_tfidf_rl import TfidfRlEngine  # noqa: E402
from src.eval.run_eval import evaluate, load_datasets, prepare_train_data  # noqa: E402

LABELED_PATH = PROCESSED_DIR / "labeled.csv"
SEED_PATH = PROCESSED_DIR / "seed.csv"
RESULTS_PATH = REPORTS_DIR / "results_fase2.csv"
N_BOOT = 2000
HARDWARE = (
    "Intel Celeron N4120 @ 1.10 GHz, 8 GB RAM (7.77 GB utilizables), "
    "Intel UHD Graphics 600 (integrada), SSD 238 GB"
)

print(
    f"Python {platform.python_version()} | pandas {pd.__version__} | "
    f"numpy {np.__version__} | scikit-learn {sklearn.__version__}"
)
print(f"Semilla: {SEED} | remuestreos bootstrap: {N_BOOT}")
print(f"Hardware: {HARDWARE}")

Python 3.12.0 | pandas 3.1.0 | numpy 2.5.4 | scikit-learn 1.9.1
Semilla: 42 | remuestreos bootstrap: 2000
Hardware: Intel Celeron N4120 @ 1.10 GHz, 8 GB RAM (7.77 GB utilizables), Intel UHD Graphics 600 (integrada), SSD 238 GB


## 1. Verificación previa de los datos

Solo revisa el esquema y la coherencia de las etiquetas, y cuenta filas. No ejecuta ningún motor ni mira predicciones.

In [7]:
problems: list[str] = []

# Lectura cruda como texto: el cargador del arnés convierte en False cualquier valor
# distinto de "True" (por ejemplo "VERDADERO"), así que se revisa antes de cargar.
for path in (LABELED_PATH, SEED_PATH):
    raw = pd.read_csv(path, encoding="utf-8", dtype=str, keep_default_na=False)
    bad_values = set(raw["should_escalate"].str.strip()) - {"True", "False"}
    if bad_values:
        problems.append(
            f"{path.name}: should_escalate con valores no válidos {sorted(bad_values)}"
        )
    dup_ids = raw.loc[raw["question_id"].duplicated(), "question_id"].tolist()
    if dup_ids:
        problems.append(f"{path.name}: question_id repetidos {dup_ids[:10]}")

labeled = load_datasets([LABELED_PATH])
seed = load_datasets([SEED_PATH])

for name, df, expected_source in (
    (LABELED_PATH.name, labeled, "real"),
    (SEED_PATH.name, seed, "synthetic"),
):
    sources = set(df["source"].astype(str).str.strip())
    if sources != {expected_source}:
        problems.append(
            f"{name}: se esperaba source='{expected_source}' y se encontró {sorted(sources)}"
        )
    splits = set(df["split"].astype(str).str.strip())
    if not splits <= {"train", "test"}:
        problems.append(f"{name}: split con valores fuera de train/test: {sorted(splits)}")
    unknown = set(df["intent"].astype(str).str.strip()) - set(INTENTS)
    if unknown:
        problems.append(f"{name}: intenciones fuera de la taxonomía: {sorted(unknown)}")
    expected_escalate = df["intent"].astype(str).str.strip().isin(ESCALATING_INTENTS)
    inconsistent = df.loc[
        expected_escalate != df["should_escalate"].astype(bool), "question_id"
    ].tolist()
    if inconsistent:
        problems.append(
            f"{name}: should_escalate no coincide con la intención en {inconsistent[:10]}"
        )
    if (df["question_text"].astype(str).str.strip() == "").any():
        problems.append(f"{name}: hay preguntas con texto vacío")

# Conjuntos de entrenamiento de las tres variantes de M1.
VARIANT_FRAMES = {
    "synthetic": seed,
    "real": labeled,
    "both": pd.concat([seed, labeled], ignore_index=True),
}

train_counts = {}
for variant, frame in VARIANT_FRAMES.items():
    train_rows = prepare_train_data(frame, train_source=variant)
    counts = train_rows["intent"].astype(str).str.strip().value_counts()
    train_counts[variant] = counts.reindex(list(INTENTS), fill_value=0)
    too_few = counts[counts < 2].index.tolist()
    if too_few:
        problems.append(
            f"Train '{variant}': intenciones con menos de 2 ejemplos {too_few}; "
            "M1 no puede hacer la validación cruzada estratificada"
        )
    if len(counts) < 2:
        problems.append(f"Train '{variant}': se necesitan al menos 2 intenciones distintas")
    absent = [intent for intent in INTENTS if train_counts[variant][intent] == 0]
    if absent:
        print(f"AVISO: el train '{variant}' no tiene ejemplos de {absent}; M1 nunca las predecirá.")

train_table = pd.DataFrame(train_counts)
train_table.loc["TOTAL"] = train_table.sum()

split_table = (
    labeled.groupby(["intent", "split"]).size().unstack(fill_value=0)
    .reindex(index=list(INTENTS), columns=["train", "test"], fill_value=0)
)

test_real = labeled.loc[labeled["split"].astype(str).str.strip() == "test"]
n_test = len(test_real)
n_should_escalate = int(test_real["should_escalate"].astype(bool).sum())
n_respondable = n_test - n_should_escalate
print(f"Test real: n = {n_test} | deben escalar = {n_should_escalate} | respondibles = {n_respondable}")
if min(n_should_escalate, n_respondable) < 10:
    print("AVISO: un subgrupo del test tiene menos de 10 filas; sus IC serán muy anchos.")
empty_in_test = [i for i in INTENTS if split_table.loc[i, "test"] == 0]
if empty_in_test:
    print(f"AVISO: sin filas de test para {empty_in_test}; esas filas de la matriz de confusión quedarán en 0.")

print("\nlabeled.csv: filas por intención y partición")
display(split_table)
print("Filas de train por intención en cada variante de entrenamiento")
display(train_table)

if problems:
    raise ValueError("Verificación previa fallida:\n- " + "\n- ".join(problems))
print("Verificación previa: sin problemas.")

Test real: n = 68 | deben escalar = 59 | respondibles = 9
AVISO: un subgrupo del test tiene menos de 10 filas; sus IC serán muy anchos.

labeled.csv: filas por intención y partición


split,train,test
intent,,
precio,18,8
capacidad,4,2
reglas_servicios,8,3
ubicacion,10,4
disponibilidad,55,23
otro_negocio,61,26
fuera_de_alcance,6,2


Filas de train por intención en cada variante de entrenamiento


,synthetic,real,both
intent,,,
precio,7,18,25
capacidad,8,4,12
reglas_servicios,8,8,16
ubicacion,8,10,18
disponibilidad,9,55,64
otro_negocio,8,61,69
fuera_de_alcance,8,6,14
TOTAL,56,162,218


Verificación previa: sin problemas.


## 2. Corrida única sobre el test real

Se llama al arnés cuatro veces (M0 y M1 con cada variante de entrenamiento). Todo se calcula en memoria y los archivos se escriben al final; `results_fase2.csv` se escribe de último, así que su existencia indica que la corrida terminó.

Además de lo que calcula el arnés, esta celda agrega:
- IC bootstrap de la mediana y del percentil 95 de la latencia.
- IC bootstrap **pareado** de la diferencia entre cada par de corridas (mismas preguntas), para saber si una diferencia es distinguible del azar.
- El `C`, el umbral y los pliegues que eligió M1 en cada variante.

Para repetir la corrida hay que borrar `results_fase2.csv` a propósito. No hacerlo para ajustar nada con el *test*.

In [8]:
if RESULTS_PATH.exists():
    raise FileExistsError(
        f"{RESULTS_PATH} ya existe: el test real se evalúa una sola vez. "
        "Bórralo a propósito solo si la corrida anterior falló antes de terminar."
    )


def bootstrap_stat_ci(values, stat, n_boot=N_BOOT, seed=SEED, confidence=0.95):
    """IC bootstrap (percentiles) de un estadístico ``stat(matriz, axis=1)``."""
    arr = np.asarray(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, arr.size, size=(n_boot, arr.size))
    stats = stat(arr[idx], axis=1)
    alpha = (1.0 - confidence) / 2.0
    return (
        float(np.percentile(stats, 100.0 * alpha)),
        float(np.percentile(stats, 100.0 * (1.0 - alpha))),
    )


def paired_bootstrap_diff(hits_a, hits_b, n_boot=N_BOOT, seed=SEED, confidence=0.95):
    """Diferencia de medias (a - b) con IC bootstrap pareado sobre las mismas preguntas."""
    a = np.asarray(hits_a, dtype=float)
    b = np.asarray(hits_b, dtype=float)
    if a.size != b.size:
        raise ValueError("Los vectores a comparar deben tener la misma longitud.")
    if a.size == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, a.size, size=(n_boot, a.size))
    diffs = a[idx].mean(axis=1) - b[idx].mean(axis=1)
    alpha = (1.0 - confidence) / 2.0
    return (
        float(a.mean() - b.mean()),
        float(np.percentile(diffs, 100.0 * alpha)),
        float(np.percentile(diffs, 100.0 * (1.0 - alpha))),
    )


m1_instances: dict[str, TfidfRlEngine] = {}


def make_m1_factory(variant: str):
    """Fábrica que guarda la instancia de M1 para leer luego C, umbral y pliegues."""

    def factory() -> TfidfRlEngine:
        engine = TfidfRlEngine()
        m1_instances[variant] = engine
        return engine

    return factory


RUNS = (
    ("M0", None),
    ("M1", "synthetic"),
    ("M1", "real"),
    ("M1", "both"),
)

summaries: list[pd.DataFrame] = []
predictions: dict[str, pd.DataFrame] = {}
matrices: dict[str, pd.DataFrame] = {}

for engine_name, variant in RUNS:
    if engine_name == "M0":
        tag = "M0"
        result = evaluate(
            eval_df=labeled,
            engine_names=["M0"],
            split="test",
            allow_test=True,
            train_df=labeled,
            train_source="real",  # M0 no usa el train; se rotula "none" más abajo
            hardware=HARDWARE,
            n_boot=N_BOOT,
        )
        train_label = "none"
    else:
        tag = f"M1_train-{variant}"
        result = evaluate(
            eval_df=labeled,
            engine_names=["M1"],
            split="test",
            allow_test=True,
            train_df=VARIANT_FRAMES[variant],
            train_source=variant,
            hardware=HARDWARE,
            n_boot=N_BOOT,
            engine_registry={"M1": make_m1_factory(variant)},
        )
        train_label = variant

    summary_df = result.summary_df.copy()
    pred_df = result.predictions_df.copy()
    summary_df["train_source"] = train_label
    pred_df["train_source"] = train_label

    latencies = pred_df["latency_ms"].astype(float).to_numpy()
    med_low, med_high = bootstrap_stat_ci(latencies, lambda a, axis: np.median(a, axis=axis))
    p95_low, p95_high = bootstrap_stat_ci(
        latencies, lambda a, axis: np.percentile(a, 95, axis=axis)
    )
    summary_df["latency_median_ci_low"] = med_low
    summary_df["latency_median_ci_high"] = med_high
    summary_df["latency_p95_ci_low"] = p95_low
    summary_df["latency_p95_ci_high"] = p95_high
    summary_df["n_boot"] = N_BOOT
    summary_df.insert(0, "run", tag)

    summaries.append(summary_df)
    predictions[tag] = pred_df
    matrices[tag] = result.confusion_matrices[(engine_name, "real", "test")]

summary = pd.concat(summaries, ignore_index=True)

# Las tres corridas deben haber visto exactamente las mismas preguntas en el mismo orden.
reference_ids = predictions["M0"]["question_id"].tolist()
for tag, pred_df in predictions.items():
    if pred_df["question_id"].tolist() != reference_ids:
        raise RuntimeError(f"La corrida {tag} no evaluó las mismas preguntas que M0.")

# Aciertos por pregunta para las comparaciones pareadas.
should_esc = predictions["M0"]["should_escalate"].to_numpy(dtype=bool)
metric_hits = {
    tag: {
        "intent_accuracy": (p["intent"] == p["predicted_intent"]).to_numpy(dtype=float),
        "correct_abstention": p.loc[should_esc, "escalate"].to_numpy(dtype=float),
        "unnecessary_abstention": p.loc[~should_esc, "escalate"].to_numpy(dtype=float),
    }
    for tag, p in predictions.items()
}

comparison_rows = []
for tag_a, tag_b in combinations(predictions, 2):
    for metric in ("intent_accuracy", "correct_abstention", "unnecessary_abstention"):
        diff, low, high = paired_bootstrap_diff(
            metric_hits[tag_a][metric], metric_hits[tag_b][metric]
        )
        comparison_rows.append(
            {
                "metric": metric,
                "run_a": tag_a,
                "run_b": tag_b,
                "n": len(metric_hits[tag_a][metric]),
                "diff_a_minus_b": diff,
                "ci_low": low,
                "ci_high": high,
                "ci_includes_zero": bool(low <= 0.0 <= high) if not np.isnan(diff) else None,
            }
        )
comparisons = pd.DataFrame(comparison_rows)

# Lo que M1 eligió en cada variante, con los valores de la validación cruzada.
selection_rows = []
for variant, engine in m1_instances.items():
    table = engine.get_threshold_tradeoff_table()
    chosen = table.loc[np.isclose(table["threshold"], engine.threshold)].iloc[0]
    selection_rows.append(
        {
            "train_source": variant,
            "n_train": len(prepare_train_data(VARIANT_FRAMES[variant], train_source=variant)),
            "cv_folds": engine.cv_folds_,
            "best_c": engine.best_c_,
            "threshold": engine.threshold,
            "cv_correct_abstention": chosen["correct_abstention"],
            "cv_unnecessary_abstention": chosen["unnecessary_abstention"],
            "cv_youden_j": chosen["youden_j"],
        }
    )
selection = pd.DataFrame(selection_rows)

RESULT_COLUMNS = [
    "run", "engine", "train_source", "source", "split",
    "n_total", "n_should_escalate", "n_respondable",
    "intent_accuracy", "intent_accuracy_ci_low", "intent_accuracy_ci_high", "intent_accuracy_ci_status",
    "correct_abstention", "correct_abstention_ci_low", "correct_abstention_ci_high", "correct_abstention_ci_status",
    "unnecessary_abstention", "unnecessary_abstention_ci_low", "unnecessary_abstention_ci_high",
    "unnecessary_abstention_ci_status",
    "latency_median_ms", "latency_median_ci_low", "latency_median_ci_high",
    "latency_p95_ms", "latency_p95_ci_low", "latency_p95_ci_high",
    "n_boot", "hardware",
]
summary_out = summary[RESULT_COLUMNS]


def save_confusion_png(matrix: pd.DataFrame, title: str, path: Path) -> bool:
    """Guarda la matriz de confusión como PNG; devuelve False si no hay matplotlib."""
    try:
        from matplotlib.figure import Figure
    except ImportError:
        return False
    values = matrix.to_numpy()
    fig = Figure(figsize=(7.5, 6.5))
    ax = fig.subplots()
    image = ax.imshow(values, cmap="Blues")
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.set_xlabel("Intención predicha")
    ax.set_ylabel("Intención real")
    ax.set_title(title)
    limit = values.max() / 2 if values.size and values.max() > 0 else 0
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, int(values[i, j]), ha="center", va="center",
                    color="white" if values[i, j] > limit else "black")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    return True


REPORTS_DIR.mkdir(parents=True, exist_ok=True)
saved_files: list[Path] = []
png_ok = True
for tag, matrix in matrices.items():
    csv_path = REPORTS_DIR / f"confusion_matrix_fase2_{tag}.csv"
    matrix.to_csv(csv_path, encoding="utf-8")
    saved_files.append(csv_path)
    png_path = REPORTS_DIR / f"confusion_matrix_fase2_{tag}.png"
    n_rows = int(matrix.to_numpy().sum())
    if save_confusion_png(matrix, f"{tag} - test real (n={n_rows})", png_path):
        saved_files.append(png_path)
    else:
        png_ok = False

selection_path = REPORTS_DIR / "results_fase2_m1_seleccion.csv"
selection.to_csv(selection_path, index=False, encoding="utf-8")
comparisons_path = REPORTS_DIR / "results_fase2_comparaciones.csv"
comparisons.to_csv(comparisons_path, index=False, encoding="utf-8")
saved_files += [selection_path, comparisons_path]

summary_out.to_csv(RESULTS_PATH, index=False, encoding="utf-8")  # de último
saved_files.append(RESULTS_PATH)

print("Archivos guardados:")
for saved in saved_files:
    print(f"  {saved.relative_to(ROOT)}")
if not png_ok:
    print("AVISO: matplotlib no está instalado; solo se guardaron las matrices en CSV.")

ADVERTENCIA: evaluación sobre test: un solo run
ADVERTENCIA: evaluación sobre test: un solo run
ADVERTENCIA: evaluación sobre test: un solo run
AVISO M1: Algunas clases tienen menos de 5 ejemplos en train ({'capacidad': 4}); se reduce el número de pliegues de validación cruzada a 4.


C:\Users\yaiza\Documents\2026B\SISTEMAS INTELIGENTES\COURSE PROJECT\chatbot-Villa-Yaiza-RAG\src\eval\run_eval.py:301: UserWarning: AVISO M1: Algunas clases tienen menos de 5 ejemplos en train ({'capacidad': 4}); se reduce el número de pliegues de validación cruzada a 4.
  engine.fit(fit_df.copy())


ADVERTENCIA: evaluación sobre test: un solo run
Archivos guardados:
  reports\confusion_matrix_fase2_M0.csv
  reports\confusion_matrix_fase2_M0.png
  reports\confusion_matrix_fase2_M1_train-synthetic.csv
  reports\confusion_matrix_fase2_M1_train-synthetic.png
  reports\confusion_matrix_fase2_M1_train-real.csv
  reports\confusion_matrix_fase2_M1_train-real.png
  reports\confusion_matrix_fase2_M1_train-both.csv
  reports\confusion_matrix_fase2_M1_train-both.png
  reports\results_fase2_m1_seleccion.csv
  reports\results_fase2_comparaciones.csv
  reports\results_fase2.csv


## 3. Resultados

Los valores van como `punto [IC 95 %]`. Un IC ancho significa que con tan pocas preguntas no se puede afirmar mucho.

In [9]:
def fmt(point: float, low: float, high: float, digits: int = 2) -> str:
    """Formatea ``punto [bajo–alto]``; devuelve n/a si no hay dato."""
    if pd.isna(point):
        return "n/a"
    return f"{point:.{digits}f} [{low:.{digits}f}–{high:.{digits}f}]"


view = pd.DataFrame(
    {
        "corrida": summary["run"],
        "n test": summary["n_total"],
        "n deben escalar": summary["n_should_escalate"],
        "n respondibles": summary["n_respondable"],
        "accuracy intención": summary.apply(
            lambda r: fmt(r["intent_accuracy"], r["intent_accuracy_ci_low"], r["intent_accuracy_ci_high"]), axis=1
        ),
        "abstención correcta": summary.apply(
            lambda r: fmt(r["correct_abstention"], r["correct_abstention_ci_low"], r["correct_abstention_ci_high"]), axis=1
        ),
        "abstención innecesaria": summary.apply(
            lambda r: fmt(r["unnecessary_abstention"], r["unnecessary_abstention_ci_low"], r["unnecessary_abstention_ci_high"]), axis=1
        ),
        "latencia mediana ms": summary.apply(
            lambda r: fmt(r["latency_median_ms"], r["latency_median_ci_low"], r["latency_median_ci_high"], 3), axis=1
        ),
        "latencia p95 ms": summary.apply(
            lambda r: fmt(r["latency_p95_ms"], r["latency_p95_ci_low"], r["latency_p95_ci_high"], 3), axis=1
        ),
    }
)
print(f"Hardware: {HARDWARE}\n")
display(view)

flags = summary.loc[
    (summary["intent_accuracy_ci_status"] != "ok")
    | (summary["correct_abstention_ci_status"] != "ok")
    | (summary["unnecessary_abstention_ci_status"] != "ok"),
    "run",
].tolist()
if flags:
    print(f"AVISO: muestra pequeña (n < 10) en alguna métrica de {flags}.")

print("\nLo que eligió M1 en cada variante (valores de la validación cruzada sobre el train)")
display(selection.round(3))

print("\nDiferencias entre corridas (a − b), IC bootstrap pareado. Si el IC incluye 0, no se distingue del azar.")
display(comparisons.round(3))

Hardware: Intel Celeron N4120 @ 1.10 GHz, 8 GB RAM (7.77 GB utilizables), Intel UHD Graphics 600 (integrada), SSD 238 GB



,corrida,n test,n deben escalar,n respondibles,accuracy intención,abstención correcta,abstención innecesaria,latencia mediana ms,latencia p95 ms
0,M0,68,59,9,0.66 [0.56–0.78],0.90 [0.81–0.97],0.33 [0.00–0.67],0.785 [0.630–1.175],2.952 [2.465–7.608]
1,M1_train-synthetic,68,59,9,0.38 [0.28–0.50],0.80 [0.69–0.90],0.22 [0.00–0.56],3.373 [2.956–3.871],35.624 [17.251–70.532]
2,M1_train-real,68,59,9,0.71 [0.60–0.81],1.00 [1.00–1.00],0.78 [0.44–1.00],3.924 [3.600–4.628],8.591 [6.568–10.071]
3,M1_train-both,68,59,9,0.74 [0.63–0.84],0.98 [0.95–1.00],0.56 [0.22–0.89],3.109 [2.962–3.355],6.382 [4.952–7.618]


AVISO: muestra pequeña (n < 10) en alguna métrica de ['M0', 'M1_train-synthetic', 'M1_train-real', 'M1_train-both'].

Lo que eligió M1 en cada variante (valores de la validación cruzada sobre el train)


,train_source,n_train,cv_folds,best_c,threshold,cv_correct_abstention,cv_unnecessary_abstention,cv_youden_j
0,synthetic,56,5,10.0,0.25,0.906,0.625,0.281
1,real,162,4,100.0,0.40,0.986,0.818,0.168
2,both,218,5,100.0,0.15,0.977,0.500,0.477



Diferencias entre corridas (a − b), IC bootstrap pareado. Si el IC incluye 0, no se distingue del azar.


,metric,run_a,run_b,n,diff_a_minus_b,ci_low,ci_high,ci_includes_zero
0,intent_accuracy,M0,M1_train-synthetic,68,0.279,0.147,0.412,False
1,correct_abstention,M0,M1_train-synthetic,59,0.102,0.000,0.220,True
2,unnecessary_abstention,M0,M1_train-synthetic,9,0.111,0.000,0.333,True
3,intent_accuracy,M0,M1_train-real,68,-0.044,-0.147,0.059,True
4,correct_abstention,M0,M1_train-real,59,-0.102,-0.186,-0.034,False
5,unnecessary_abstention,M0,M1_train-real,9,-0.444,-0.889,0.000,True
6,intent_accuracy,M0,M1_train-both,68,-0.074,-0.177,0.044,True
7,correct_abstention,M0,M1_train-both,59,-0.085,-0.169,0.000,True
8,unnecessary_abstention,M0,M1_train-both,9,-0.222,-0.669,0.333,True
9,intent_accuracy,M1_train-synthetic,M1_train-real,68,-0.324,-0.471,-0.176,False


In [10]:
for tag, matrix in matrices.items():
    print(f"\nMatriz de confusión: {tag} (filas = intención real, columnas = predicha)")
    display(matrix)


Matriz de confusión: M0 (filas = intención real, columnas = predicha)


,precio,capacidad,reglas_servicios,ubicacion,disponibilidad,otro_negocio,fuera_de_alcance
true_intent,,,,,,,
precio,4,2,0,0,0,2,0
capacidad,0,2,0,0,0,0,0
reglas_servicios,0,1,0,1,0,1,0
ubicacion,0,0,0,2,0,2,0
disponibilidad,0,0,0,0,20,3,0
otro_negocio,2,1,1,2,3,17,0
fuera_de_alcance,0,0,0,0,0,2,0



Matriz de confusión: M1_train-synthetic (filas = intención real, columnas = predicha)


,precio,capacidad,reglas_servicios,ubicacion,disponibilidad,otro_negocio,fuera_de_alcance
true_intent,,,,,,,
precio,4,2,2,0,0,0,0
capacidad,0,2,0,0,0,0,0
reglas_servicios,0,1,1,1,0,0,0
ubicacion,0,0,1,2,0,1,0
disponibilidad,11,4,0,0,8,0,0
otro_negocio,4,3,5,3,1,9,1
fuera_de_alcance,1,0,0,1,0,0,0



Matriz de confusión: M1_train-real (filas = intención real, columnas = predicha)


,precio,capacidad,reglas_servicios,ubicacion,disponibilidad,otro_negocio,fuera_de_alcance
true_intent,,,,,,,
precio,5,0,0,0,2,1,0
capacidad,0,0,0,0,2,0,0
reglas_servicios,1,0,0,0,1,1,0
ubicacion,0,0,0,2,0,2,0
disponibilidad,0,0,0,0,19,4,0
otro_negocio,1,0,0,0,2,22,1
fuera_de_alcance,1,0,0,0,1,0,0



Matriz de confusión: M1_train-both (filas = intención real, columnas = predicha)


,precio,capacidad,reglas_servicios,ubicacion,disponibilidad,otro_negocio,fuera_de_alcance
true_intent,,,,,,,
precio,5,0,1,0,2,0,0
capacidad,0,0,0,0,2,0,0
reglas_servicios,1,0,1,0,1,0,0
ubicacion,0,0,0,3,0,1,0
disponibilidad,0,0,0,0,19,4,0
otro_negocio,1,0,0,0,2,22,1
fuera_de_alcance,1,0,0,0,1,0,0


## 4. Notas para el reporte

- Es **una sola corrida** sobre el *test* real del lote 1; no se ajustó nada con él.
- Solo hay preguntas reales: la comparación sobre preguntas sintéticas queda pendiente.
- El tamaño del *test* es pequeño. Cuando el IC de una diferencia incluye 0, la diferencia no es concluyente aunque el punto favorezca a un motor.
- Las etiquetas del lote las puso una sola persona, sin doble etiquetado: el error de etiqueta no está medido.
- La latencia se midió dentro de `respond()` (sin carga del modelo) en el equipo indicado en la columna `hardware`. No es comparable con latencias medidas en otro computador.
- La abstención correcta e innecesaria depende de cuántas preguntas haya en cada subgrupo (`n deben escalar` y `n respondibles`).